In [ ]:
import data_utils
import requests
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import Counter
import random
import os
from tqdm import tqdm
from huggingface_hub import login

In [ ]:
login(token="your_huggingface_token")
print("Login successful!")

In [ ]:
dataset = 'imagenet'

d_train = dataset + "_train"
data = data_utils.get_data(d_train)

os.makedirs("data/generated_sentence", exist_ok=True)

cls_file = data_utils.LABEL_FILES[dataset]
with open(cls_file, "r") as f:
    classes = f.read().split("\n")

In [ ]:
data_dict = Counter(data.targets)
counter = [0]
for i in range(len(classes)):
    counter.append(counter[i] + data_dict[i])

In [ ]:
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
for i in tqdm(range(140, len(classes))):
    samples = random.sample(range(counter[i], counter[i+1]), 50)
    for sample in samples:
        img, label = data[sample]
        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": "If I had to describe this one using only one sentence with the words {}, it would be: ".format(classes[label])}
            ]}
        ]
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(
            img,
            input_text,
            add_special_tokens=False,
            return_tensors="pt"
        ).to(model.device)

        output = model.generate(**inputs, max_new_tokens=30)
        with open("data/generated_sentence/{}_generated_sentences.txt".format(dataset), "a", encoding="utf-8") as file:
            file.write(processor.decode(output[0]).split('\n')[-1].replace("<|eot_id|>", "") + "\n")